# Wan 2.2 Image-to-Video — Official Repository (fully automatic, T4 Colab)

Built on the **official [`Wan-Video/Wan2.2`](https://github.com/Wan-Video/Wan2.2) GitHub repository** — native inference (`generate.py`), not the `diffusers` integration. `AutoPipelineForImage2Video` is never imported anywhere in this notebook.

**Runtime:** `Runtime -> Change runtime type -> T4 GPU`, then `Runtime -> Run all`. The only thing you'll be asked for is the product image in Step 5 — every other cell runs unattended, in order, with no manual restart.

**Why there's no restart step:** installing this repo's dependencies only adds packages Colab's own bootstrap never imports for itself (`diffusers`, `transformers`, `accelerate`, `easydict`, `dashscope`, …) — the install cell deliberately leaves `torch`, `torchvision`, `torchaudio` **and** `numpy` exactly as Colab shipped them, since those are the packages Colab's own internals may already have loaded, and reinstalling them mid-session is what actually forces the restart-and-lose-your-place problem most Wan 2.2 Colab notebooks hit. Nothing in this notebook imports any of those packages before the install cell runs, so there's nothing already in memory to go stale.

**Model:** `Wan-AI/Wan2.2-TI2V-5B`, run via the official `ti2v-5B` task — the only Wan 2.2 checkpoint the repo positions for a single consumer-class GPU. The 14B `T2V-A14B` / `I2V-A14B` models need **at least 80GB VRAM** per the official README, so they're not used here.

**On the output size:** the repo hard-codes which sizes each task accepts (`wan/configs/__init__.py`'s `SUPPORTED_SIZES`) — `480*832` is only valid for the 80GB+ tasks; passing it to `ti2v-5B` raises `AssertionError`, confirmed by reading that file directly. `ti2v-5B`'s only 9:16 option is `704*1280`, used below.

**Automatic T4 fallback:** the official README states TI2V-5B needs at least 24GB VRAM even with every memory-saving flag this notebook enables — a free T4 has 16GB, so a CUDA out-of-memory error is a real possibility, not a bug. Step 6 handles this itself: if generation runs out of memory, it automatically retries with a shorter clip (5s → 3s → 2s → 1s) instead of just crashing, since a free T4's exact headroom varies session to session.

**`flash_attn` is intentionally skipped** (also filtered out of the install). Its own attention module (`wan/modules/attention.py`) falls back to PyTorch's native `scaled_dot_product_attention` when `flash_attn` isn't installed — confirmed by reading that file directly. Compiling `flash_attn` in Colab is slow and failure-prone (the repo's own `INSTALL.md` calls this out); skipping it trades a little speed for an install that reliably finishes unattended.

## Step 1 — Confirm the GPU

In [ ]:
!nvidia-smi
print('✅ Step 1/7 done — GPU confirmed.')

## Step 2 — Clone the official Wan 2.2 repository

In [ ]:
%cd /content
!git clone --quiet https://github.com/Wan-Video/Wan2.2.git
%cd /content/Wan2.2
print('✅ Step 2/7 done — official Wan-Video/Wan2.2 repository cloned.')

## Step 3 — Install dependencies (no restart needed)

See the "Why there's no restart step" note above — `torch`, `torchvision`, `torchaudio`, `numpy` and `flash_attn` are deliberately left out of this install.

In [ ]:
!grep -viE '^(flash_attn|torch|torchvision|torchaudio|numpy)' requirements.txt > requirements_colab.txt
!pip install -q -r requirements_colab.txt
!pip install -q "huggingface_hub[cli]"

import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        'No GPU detected. Click Runtime > Change runtime type, select T4 GPU, '
        'save, then Runtime > Run all. This is the one situation this notebook '
        "can't fix by itself — Colab only assigns a GPU after you pick one."
    )

torch_version = tuple(int(x) for x in torch.__version__.split('+')[0].split('.')[:2])
if torch_version < (2, 4):
    raise RuntimeError(
        f'Wan 2.2 needs torch>=2.4.0; this runtime has {torch.__version__}. '
        'Click Runtime > Disconnect and delete runtime, then Runtime > Run all '
        "to get a fresh runtime with a current torch build — this notebook "
        'intentionally avoids reinstalling torch itself (see the note above).'
    )

print('✅ Step 3/7 done — dependencies installed, GPU + torch verified, no restart required.')

## Step 4 — Download the T4-compatible Wan 2.2 model (TI2V-5B)

Several GB from Hugging Face — first run takes a few minutes.

In [ ]:
!huggingface-cli download Wan-AI/Wan2.2-TI2V-5B --local-dir ./Wan2.2-TI2V-5B
print('✅ Step 4/7 done — Wan-AI/Wan2.2-TI2V-5B downloaded.')

## Step 5 — Upload your product image

The only manual step in this notebook — everything else runs on its own.

In [ ]:
from google.colab import files

print('Choose one product photo to upload:')
uploaded = files.upload()  # saves into the current directory: /content/Wan2.2
image_filename = next(iter(uploaded))
print(f'✅ Step 5/7 done — uploaded {image_filename}.')

## Step 6 — Generate a 9:16 vertical video (auto-retries on out-of-memory)

`704*1280` is `ti2v-5B`'s supported 9:16 size (see the size note above). Runs the official `generate.py` as a subprocess — if a run reports a CUDA out-of-memory error, this cell automatically shortens the clip and tries again (fresh subprocess each time, so GPU memory is fully released between attempts) instead of stopping.

In [ ]:
import subprocess

# --- Tunables -----------------------------------------------------------
SIZE = '704*1280'                     # ti2v-5B's only 9:16 option (480*832 is not valid for this task)
FPS = 24                              # fixed by the ti2v-5B model config (sample_fps)
DURATION_LADDER_SECONDS = [5, 3, 2, 1]  # automatic fallback order if one length hits CUDA OOM
PROMPT = 'A realistic, premium commercial product video. Natural, smooth camera motion, cinematic lighting.'  # edit me
SAVE_FILE = 'output.mp4'
# --------------------------------------------------------------------------

IMAGE_PATH = f'/content/Wan2.2/{image_filename}'

def frame_num_for(seconds):
    return int(round((FPS * seconds - 1) / 4)) * 4 + 1  # frame counts must be 4n+1

def run_generate(seconds):
    frame_num = frame_num_for(seconds)
    print(f'>>> Trying {frame_num} frames (~{frame_num / FPS:.1f}s @ {FPS}fps) at {SIZE}...')
    return subprocess.run(
        [
            'python', 'generate.py',
            '--task', 'ti2v-5B',
            '--size', SIZE,
            '--ckpt_dir', './Wan2.2-TI2V-5B',
            '--offload_model', 'True',
            '--convert_model_dtype',
            '--t5_cpu',
            '--image', IMAGE_PATH,
            '--prompt', PROMPT,
            '--frame_num', str(frame_num),
            '--save_file', SAVE_FILE,
        ],
        capture_output=True,
        text=True,
    )

succeeded = False
for attempt_index, seconds in enumerate(DURATION_LADDER_SECONDS):
    proc = run_generate(seconds)

    if proc.returncode == 0:
        print(f'✅ Step 6/7 done — generated a ~{seconds}s clip at {SIZE}: {SAVE_FILE}')
        succeeded = True
        break

    stderr_tail = (proc.stderr or '')[-4000:]
    is_oom = 'out of memory' in stderr_tail.lower()
    is_last_attempt = attempt_index == len(DURATION_LADDER_SECONDS) - 1

    if is_oom and not is_last_attempt:
        print(f'⚠️  CUDA out of memory at ~{seconds}s — automatically retrying with a shorter clip...')
        continue

    print(stderr_tail)
    reason = 'CUDA out of memory even at the shortest fallback length' if is_oom else 'generate.py failed'
    raise RuntimeError(
        f'{reason}. This T4 session may simply have less free VRAM than usual right now — '
        'Runtime > Disconnect and delete runtime, then Runtime > Run all often gets you a session '
        'with more headroom (Colab does not let a notebook request a specific one). '
        'Full error above.'
    )

assert succeeded

## Step 7 — Preview and automatically download the MP4

In [ ]:
from IPython.display import Video, display
display(Video(SAVE_FILE, embed=True))

files.download("output.mp4")
print('✅ Step 7/7 done — output.mp4 downloaded.')